# 08 · Live demo — Gradio → SGLang
**Purpose:** paste an abstract + research question; the SGLang-served model D (curated) answers with grounded reasoning and a yes/no/maybe label, side by side with model B (raw).
Examples are from DEV (never TEST). **Inputs:** `artifacts/models/{B,D}_merged/`, `artifacts/data/dev.jsonl`. **Outputs:** a public Gradio link.
**Runtime (T4):** ~3 min to start both servers.

In [ ]:
import os, sys, json
if not os.path.ismount("/content/drive"):
    from google.colab import drive; drive.mount("/content/drive")
sys.path.insert(0, "/content/drive/MyDrive/ps8/repo/src")
import ps8_common as C
C.ensure_env()
cfg = C.load_config()
GPU, P = C.gpu_profile(cfg)
SIDE_BY_SIDE = True

In [ ]:
# Two 1.5B models fit on one T4 at 0.4 memory fraction each. start_server stops anything on its own port only.
C.stop_server(30000); C.stop_server(30001)
C.start_server(str(C.ART / "models" / "D_merged"), "D", P["sglang_args"], port=30000, mem_fraction=0.4)
if SIDE_BY_SIDE:
    # start_server tracks one process; launch B second so D keeps running (stop both with pkill at the end).
    import ps8_common
    d_proc, ps8_common._SERVER = ps8_common._SERVER, None
    C.start_server(str(C.ART / "models" / "B_merged"), "B", P["sglang_args"], port=30001, mem_fraction=0.4)

In [ ]:
import gradio as gr
dev = C.read_jsonl(C.ART / "data" / "dev.jsonl")
fewshot_ids = {r["pubid"] for r in json.load(open(C.DATA / "fewshot.json"))}
examples = []
for lab in ("yes", "no", "maybe"):
    r = next(r for r in dev if r["final_decision"] == lab and r["pubid"] not in fewshot_ids and len(r["context_text"].split()) < 250)
    examples.append([r["context_text"], r["question"]])


def answer(abstract, question):
    msgs = C.build_messages(abstract.strip(), question.strip())
    d = C.chat(msgs, "D", base_url="http://localhost:30000/v1", max_tokens=cfg["eval"]["max_tokens"])
    out = [d["text"], f"**{(C.parse_answer(d['text']) or 'unparseable').upper()}** · {d['latency_s']:.2f}s"]
    if SIDE_BY_SIDE:
        b = C.chat(msgs, "B", base_url="http://localhost:30001/v1", max_tokens=cfg["eval"]["max_tokens"])
        out += [b["text"], f"**{(C.parse_answer(b['text']) or 'unparseable').upper()}** · {b['latency_s']:.2f}s"]
    return out


with gr.Blocks(title="MedCurate — PubMedQA") as demo:
    gr.Markdown("## MedCurate · curated-synthetic-data PubMedQA assistant\nQwen2.5-1.5B fine-tuned (AlignTune) on CuratorKIT-curated synthetic QA, served by **SGLang**. Answers only from the abstract.")
    abstract = gr.Textbox(label="Abstract", lines=10)
    question = gr.Textbox(label="Research question")
    go = gr.Button("Answer", variant="primary")
    with gr.Row():
        with gr.Column():
            gr.Markdown("### D · trained on CURATED data")
            d_txt, d_lab = gr.Textbox(label="Reasoning", lines=5), gr.Markdown()
        if SIDE_BY_SIDE:
            with gr.Column():
                gr.Markdown("### B · trained on RAW generated data")
                b_txt, b_lab = gr.Textbox(label="Reasoning", lines=5), gr.Markdown()
    outputs = [d_txt, d_lab] + ([b_txt, b_lab] if SIDE_BY_SIDE else [])
    go.click(answer, [abstract, question], outputs)
    gr.Examples(examples, [abstract, question])

demo.launch(share=True, prevent_thread_lock=True)

In [ ]:
# When done: demo.close(); then free the GPU.
# demo.close(); import subprocess; subprocess.run("pkill -f sglang.launch_server", shell=True)